# Part C — Group investigation

**This is the notebook your group submits.** Rename it with your group
identifier. Keep the numerical outputs that support your conclusions,
but do not embed large GIF or video files.

## Assessment

| Criterion | Weight |
|---|---:|
| Focused question, prediction, and controlled design | 25% |
| Appropriate quantitative evidence | 30% |
| Physical interpretation | 25% |
| Limitations and reproducibility | 10% |
| Clear notebook and contribution statement | 10% |

Use one main independent variable. A two-factor study requires instructor
approval. Your notebook must run from top to bottom before submission.


## Group and research question

**Group members:**  

**Research question:**  

**Why this question matters for long surface waves:**


## Prediction

State your expected result **before** presenting simulation results. Name
the physical mechanism or scaling that supports it.

**Prediction:**


## Model scope

This one-layer shallow-water model is hydrostatic and depth averaged. In
the default linear configuration it assumes small surface displacement.
All cells remain wet; it does not represent breaking, run-up, inundation,
or a resolved bottom boundary layer. Rayleigh damping is an idealized
energy-loss parameter.


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from shallowwater import (
    ModelParams, backend_info, compute_dt_cfl, depth_on_u,
    load_bathymetry, make_grid, make_wind_forcing_from_file,
    run_model, shelf_bathymetry, uniform_wind_forcing, zero_forcing,
)

print(backend_info())


## Reusable experiment function

The function below makes the controls explicit. `depth` may be a positive
scalar, an `(Ny, Nx)` array, or a function that creates such an array from
the grid. Initial-state choices are `cross_pulse`, `gaussian`, and `rest`.
Uniform wind is controlled by `wind_x`, `wind_y`, `wind_ramp_hours`, and
`wind_off_hours`.

If you use a custom external file, the submitted notebook must explain
how it can be reproduced. Prefer generating arrays in this notebook or
using the supplied course files; do not use an unexplained absolute path.


In [ ]:
def make_initial_state(
    grid, params, *, kind="cross_pulse", amplitude=0.08,
    radius=60e3, x_fraction=0.25, y_fraction=0.50,
):
    x0, y0 = x_fraction * grid.Lx, y_fraction * grid.Ly
    eta = np.zeros((grid.Ny, grid.Nx))
    u = np.zeros((grid.Ny, grid.Nx + 1))
    v = np.zeros((grid.Ny + 1, grid.Nx))
    X, Y = np.meshgrid(grid.x_c, grid.y_c)

    if kind == "rest":
        return eta, u, v
    if kind == "gaussian":
        eta = amplitude * np.exp(-((X-x0)**2 + (Y-y0)**2) / radius**2)
        return eta, u, v
    if kind == "cross_pulse":
        eta_line = amplitude * np.exp(-((grid.x_c-x0) / radius)**2)
        eta = np.repeat(eta_line[None, :], grid.Ny, axis=0)
        H_u = depth_on_u(grid, params.H)
        eta_u = amplitude * np.exp(-((grid.x_u-x0) / radius)**2)
        u = np.repeat(eta_u[None, :], grid.Ny, axis=0) * np.sqrt(params.g/H_u)
        return eta, u, v
    raise ValueError("kind must be 'cross_pulse', 'gaussian', or 'rest'")


def run_case(
    label,
    *,
    Nx=120, Ny=24, Lx=1.2e6, Ly=240e3,
    depth=400.0, f0=0.0, damping=0.0,
    initial_kind="cross_pulse", initial_amplitude=0.08,
    initial_radius=60e3, initial_x_fraction=0.25,
    wind_x=0.0, wind_y=0.0, wind_ramp_hours=1.0,
    wind_off_hours=None, wind_file=None,
    tmax_hours=5.0,
):
    grid = make_grid(Nx, Ny, Lx, Ly)
    H = depth(grid) if callable(depth) else depth
    if isinstance(H, (str, Path)):
        H = load_bathymetry(H, grid)
    params = ModelParams(
        H=H, g=9.81, f0=f0, beta=0.0,
        r=damping, linear=True,
    )
    dt = compute_dt_cfl(grid, params, cfl=0.42)

    if wind_file is not None:
        envelope = lambda t: min(1.0, t/(wind_ramp_hours*3600))
        forcing = make_wind_forcing_from_file(wind_file, grid, envelope=envelope)
    elif wind_x != 0.0 or wind_y != 0.0:
        t_off = None if wind_off_hours is None else wind_off_hours * 3600
        forcing = lambda t, g, p: uniform_wind_forcing(
            t, g, p, tau_x=wind_x, tau_y=wind_y,
            t_ramp=wind_ramp_hours*3600, t_off=t_off,
        )
    else:
        forcing = zero_forcing

    initial = lambda g, p: make_initial_state(
        g, p, kind=initial_kind, amplitude=initial_amplitude,
        radius=initial_radius, x_fraction=initial_x_fraction,
    )
    out = run_model(
        tmax=tmax_hours*3600, dt=dt, grid=grid, params=params,
        forcing_fn=forcing, ic_fn=initial, save_every=5,
        out_vars=("eta",),
    )
    print(
        f"{label}: Nx={Nx}, Ny={Ny}, Lx={Lx/1e3:.0f} km, "
        f"dx={grid.dx/1e3:.1f} km, dt={dt:.1f} s, frames={len(out['time'])}"
    )
    return {"label": label, "grid": grid, "params": params, "out": out}


## Baseline configuration

Describe the baseline and justify the domain, depth, initial state,
forcing, duration, and resolution. The default below runs successfully;
replace it with the baseline appropriate to your question.

**Baseline justification:**


In [ ]:
baseline = run_case(
    "baseline",
    depth=400.0,
    initial_kind="cross_pulse",
    tmax_hours=5.0,
)


## Controlled variations

Add at least two cases. Change one primary parameter while keeping the
remaining controls fixed. Examples include wind duration, shelf width,
domain length, or initial radius.

**Independent variable:**  

**Values tested:**  

**Controls held fixed:**


In [ ]:
# Replace these examples with cases that answer your research question.
# variation_1 = run_case("variation 1", depth=...)
# variation_2 = run_case("variation 2", depth=...)
cases = [baseline]  # add variation_1 and variation_2


## Diagnostics

Use at least one quantitative diagnostic, not only an animation. Suitable
choices include arrival time, measured propagation speed, maximum surface
elevation at a stated location, reflection time, oscillation period, or
a comparison with a theoretical scaling.


In [ ]:
def plot_hovmoller(case):
    grid, out = case["grid"], case["out"]
    eta_line = np.asarray(out["eta"]).mean(axis=1)
    times = np.asarray(out["time"])
    fig, ax = plt.subplots(figsize=(8.5, 3.8))
    image = ax.pcolormesh(
        grid.x_c/1e3, times/3600, eta_line,
        shading="auto", cmap="RdBu_r",
    )
    ax.set(xlabel="x [km]", ylabel="time [hours]", title=case["label"])
    fig.colorbar(image, ax=ax, label="surface displacement [m]")
    plt.show()


for case in cases:
    plot_hovmoller(case)


In [ ]:
# Add your quantitative measurement and comparison table here.
summary = []
for case in cases:
    eta = np.asarray(case["out"]["eta"])
    summary.append((case["label"], float(np.max(np.abs(eta)))))

print("case | maximum absolute surface displacement [m]")
for label, value in summary:
    print(f"{label:20s} | {value:.4f}")


## Results

Present the evidence needed to answer the question. Every figure needs
labelled axes with units, a useful caption or nearby explanation, and a
statement identifying what should be noticed.

**Results and figure interpretation:**


## Comparison with theory

Compare at least one result with a relevant prediction or scaling, such
as (c=\sqrt{gH}), a crossing time (L/c), a basin period (2L/c),
linear amplitude scaling, or a force/damping timescale.

**Theoretical comparison, including units and discrepancy:**


## Limitations

Discuss at least two limitations relevant to your particular conclusion.
Distinguish limitations of the physical model from numerical resolution
or experimental-design limitations.

**Limitations:**


## Conclusion

Answer the research question directly in a short paragraph. State whether
the prediction was supported and cite the main quantitative evidence.

**Conclusion:**


## Contributions and submission check

**Member contributions:**  

Before submission:

- [ ] All group members are named.
- [ ] The notebook uses no unexplained absolute file paths.
- [ ] Required figures and numerical outputs are visible.
- [ ] Large animations have been removed.
- [ ] The kernel was restarted and all cells ran in order without error.
- [ ] The filename contains the group identifier.
